In [1]:
import numpy as np
import sounddevice as sd
import librosa
import joblib
import noisereduce as nr
from scipy import signal


In [2]:
model = joblib.load("models/xgboost.pkl")
scaler = joblib.load("models/scaler.pkl")

print("✅ Model and scaler loaded")


✅ Model and scaler loaded


In [3]:
SAMPLE_RATE = 22050
RECORD_SECONDS = 2.5  # listen duration


In [4]:
def is_silent(audio, threshold=5e-4):
    energy = np.mean(audio ** 2)
    print(f"[DEBUG] audio energy:{energy :.8f}")
    return energy < threshold


In [5]:
def extract_features(audio, sr=22050):
    try:
        # Light trimming only (no noise reduction)
        audio, _ = librosa.effects.trim(audio, top_db=15)

        if len(audio) < 2000:
            return None

        features = []

        mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
        features.extend(np.mean(mfccs, axis=1))
        features.extend(np.std(mfccs, axis=1))
        features.extend(np.max(mfccs, axis=1))

        delta = librosa.feature.delta(mfccs)
        features.extend(np.mean(delta, axis=1))
        features.extend(np.std(delta, axis=1))

        features.extend([
            np.mean(librosa.feature.spectral_centroid(y=audio, sr=sr)),
            np.mean(librosa.feature.spectral_rolloff(y=audio, sr=sr)),
            np.mean(librosa.feature.zero_crossing_rate(audio)),
            np.mean(librosa.feature.rms(y=audio))
        ])

        sos = signal.butter(4, [500, 2000], btype='band', fs=sr, output='sos')
        filtered = signal.sosfilt(sos, audio)
        features.append(np.sum(filtered**2) / (np.sum(audio**2) + 1e-8))

        return np.array(features)

    except:
        return None

In [6]:
def record_and_predict():
    input("▶ Press ENTER to start listening...")

    print("🎤 Listening...")
    audio = sd.rec(
        int(SAMPLE_RATE * RECORD_SECONDS),
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32"
    )
    sd.wait()

    audio = audio.flatten()

    # 🔊 Normalize mic loudness (CRITICAL)
    audio = audio / (np.max(np.abs(audio)) + 1e-6)

    # Silence check
    if is_silent(audio):
        print("\n========== FINAL RESULT ==========")
        print("🟢 NON-COUGH (Low energy / silence)")
        print("=================================")
        return

    feat = extract_features(audio, SAMPLE_RATE)

    if feat is None:
        print("\n========== FINAL RESULT ==========")
        print("🟢 NON-COUGH (Invalid or too short audio)")
        print("=================================")
        return

    feat = scaler.transform(feat.reshape(1, -1))
    pred = model.predict(feat)[0]

    print("\n========== FINAL RESULT ==========")
    if pred == 1:
        print("🔴 COUGH")
    else:
        print("🟢 NON-COUGH")
    print("=================================")

In [ ]:
record_and_predict() 

▶ Press ENTER to start listening... 


🎤 Listening...
